# Module 7 — Real-Time Fraud: populate the Eventhouse (`LiveTransactions`)

This notebook **replaces the setup PowerShell scripts**. From Spark it loads enriched "live"
card transactions into the RTI Eventhouse and verifies the row count. The four KQL
fraud-detection rules are **run manually in the UI** — see the walkthrough §5.3.

**Before you run:**
1. Create an **Eventhouse** named `NorthwindRTI` in this workspace (**+ New item -> Eventhouse**).
   A KQL database of the same name is created automatically.
2. Open the database -> **database details** card -> copy the **Query URI**.
3. Paste it into `KUSTO_URI` in the next cell.

**Default lakehouse:** `FabricIQ` — the source CSVs live in `Files/NorthwindLakehouse/payments`.


In [ ]:
# ---- Config: paste YOUR Eventhouse Query URI here ----------------------------------------
KUSTO_URI = "https://YOUR_KQL_CLUSTER.kusto.fabric.microsoft.com"  # <- Eventhouse -> database details -> Query URI
DATABASE  = "NorthwindRTI"
TABLE     = "LiveTransactions"
BASE      = "Files/NorthwindLakehouse/payments"   # source CSVs in the FabricIQ lakehouse
FORMAT    = "com.microsoft.kusto.spark.synapse.datasource"

In [ ]:
from pyspark.sql import functions as F

txns  = spark.read.option("header", True).csv(f"{BASE}/transactions.csv")
custs = spark.read.option("header", True).csv(f"{BASE}/customers.csv").select("customerId", "homeCountry")
merch = spark.read.option("header", True).csv(f"{BASE}/merchants.csv").select("merchantId", "riskFlag")

# all confirmed-fraud rows + a 5,000-row sample of normal traffic for realistic volume
fraud  = txns.filter(F.col("isFraud") == "1")
normal = txns.filter(F.col("isFraud") != "1").limit(5000)
sample = fraud.unionByName(normal)

# denormalize homeCountry + merchantRisk so the KQL rules are self-contained, and cast types
events = (sample
    .join(custs, "customerId", "left")
    .join(merch, "merchantId", "left")
    .select(
        "txnId", "cardId", "accountId", "customerId", "merchantId",
        F.col("amount").cast("double").alias("amount"),
        "currency",
        F.col("timestamp").cast("timestamp").alias("eventTime"),
        "channel", "geoCountry",
        F.col("mcc").cast("long").alias("mcc"),
        "homeCountry",
        (F.col("riskFlag") == "1").alias("merchantRisk"),
        (F.col("isFraud") == "1").alias("isFraud"),
        "fraudType",
    ))

print(f"events: {events.count()}  (all fraud + 5,000 normal)")
events.printSchema()

In [ ]:
import requests

def kql_mgmt(csl: str):
    """Run a KQL management command against the Eventhouse."""
    tok = notebookutils.credentials.getToken(KUSTO_URI)
    r = requests.post(f"{KUSTO_URI}/v1/rest/mgmt",
                      headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"},
                      json={"db": DATABASE, "csl": csl})
    r.raise_for_status()
    return r.json()

# make re-runs clean: drop the table so we never append duplicates
kql_mgmt(f".drop table {TABLE} ifexists")

(events.write
    .format(FORMAT)
    .option("kustoCluster", KUSTO_URI)
    .option("kustoDatabase", DATABASE)
    .option("kustoTable", TABLE)
    .option("accessToken", notebookutils.credentials.getToken(KUSTO_URI))
    .option("tableCreateOptions", "CreateIfNotExist")
    .mode("Append")
    .save())

print("write complete")

In [ ]:
def kql(query: str):
    """Run a KQL query and return the result as a Spark DataFrame."""
    return (spark.read
        .format(FORMAT)
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", DATABASE)
        .option("kustoQuery", query)
        .option("accessToken", notebookutils.credentials.getToken(KUSTO_URI))
        .load())

display(kql(f"{TABLE} | count"))